In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/gdrive')
dataset = pd.read_csv("gdrive/My Drive/Sleep_health_and_lifestyle_dataset.csv")
dataset

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
...,...,...,...,...,...,...,...,...,...,...,...,...,...
554,555,Female,43,Teacher,6.7,7,45,4,Overweight,135/90,65,6000,Insomnia
555,556,Male,43,Salesperson,6.5,6,45,7,Overweight,130/85,72,6000,Insomnia
556,557,Female,43,Teacher,6.7,7,45,4,Overweight,135/90,65,6000,Insomnia
557,558,Male,43,Salesperson,6.4,6,45,7,Overweight,130/85,72,6000,Insomnia


In [ ]:
dataset['Sleep Disorder']=dataset['Sleep Disorder'].fillna("Healthy")

In [ ]:
dataset['Systolic Pressure'] = dataset['Blood Pressure'].apply(lambda x: int(x.split('/')[0]))
dataset['Diastolic Pressure'] = dataset['Blood Pressure'].apply(lambda x: int(x.split('/')[1]))

In [ ]:
dataset.drop('Blood Pressure',inplace=True,axis=1)
dataset.drop('Person ID',inplace=True,axis=1)

In [ ]:
print(dataset['Sleep Disorder'].value_counts())

Sleep Disorder
Healthy        375
Insomnia        93
Sleep Apnea     91
Name: count, dtype: int64


In [ ]:
ctgan_data = dataset

In [ ]:
print(ctgan_data.shape)
ctgan_data.head()

(559, 13)


,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder,Systolic Pressure,Diastolic Pressure
0,Male,27,Software Engineer,6.1,6,42,6,Overweight,77,4200,Healthy,126,83
1,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,Healthy,125,80
2,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,Healthy,125,80
3,Male,28,Sales Representative,5.9,4,30,8,Obese,85,3000,Sleep Apnea,140,90
4,Male,28,Sales Representative,5.9,4,30,8,Obese,85,3000,Sleep Apnea,140,90


In [ ]:
ctgan_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 559 entries, 0 to 558
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Gender                   559 non-null    object 
 1   Age                      559 non-null    int64  
 2   Occupation               559 non-null    object 
 3   Sleep Duration           559 non-null    float64
 4   Quality of Sleep         559 non-null    int64  
 5   Physical Activity Level  559 non-null    int64  
 6   Stress Level             559 non-null    int64  
 7   BMI Category             559 non-null    object 
 8   Heart Rate               559 non-null    int64  
 9   Daily Steps              559 non-null    int64  
 10  Sleep Disorder           559 non-null    object 
 11  Systolic Pressure        559 non-null    int64  
 12  Diastolic Pressure       559 non-null    int64  
dtypes: float64(1), int64(8), object(4)
memory usage: 56.9+ KB


In [ ]:
categorical_columns = [
    'Gender',
    'Occupation',
    'BMI Category',
    'Sleep Disorder'
]

print("Categorical columns:")
print(categorical_columns)

Categorical columns:
['Gender', 'Occupation', 'BMI Category', 'Sleep Disorder']


In [ ]:
for col in categorical_columns:
    print("\n", col)
    print(ctgan_data[col].value_counts())


 Gender
Gender
Male      320
Female    239
Name: count, dtype: int64

 Occupation
Occupation
Doctor                  133
Lawyer                   94
Nurse                    80
Engineer                 75
Accountant               68
Teacher                  55
Salesperson              37
Scientist                 8
Software Engineer         6
Sales Representative      2
Manager                   1
Name: count, dtype: int64

 BMI Category
BMI Category
Normal           339
Overweight       170
Normal Weight     38
Obese             12
Name: count, dtype: int64

 Sleep Disorder
Sleep Disorder
Healthy        375
Insomnia        93
Sleep Apnea     91
Name: count, dtype: int64


In [ ]:
ctgan_train_data = ctgan_data.copy()

print(ctgan_train_data.shape)

(559, 13)


In [ ]:
ctgan_train_data.head()

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder,Systolic Pressure,Diastolic Pressure
0,Male,27,Software Engineer,6.1,6,42,6,Overweight,77,4200,Healthy,126,83
1,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,Healthy,125,80
2,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,Healthy,125,80
3,Male,28,Sales Representative,5.9,4,30,8,Obese,85,3000,Sleep Apnea,140,90
4,Male,28,Sales Representative,5.9,4,30,8,Obese,85,3000,Sleep Apnea,140,90


In [ ]:
!pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.0/207.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 8.8 MB/s eta 0:00:00


In [ ]:
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import Metadata
from sdv.sampling import Condition

In [ ]:
metadata = Metadata.detect_from_dataframe(
    data=ctgan_train_data
)

print(metadata)

{
    "tables": {
        "table": {
            "columns": {
                "Gender": {
                    "sdtype": "categorical"
                },
                "Age": {
                    "sdtype": "numerical"
                },
                "Occupation": {
                    "sdtype": "categorical"
                },
                "Sleep Duration": {
                    "sdtype": "numerical"
                },
                "Quality of Sleep": {
                    "sdtype": "categorical"
                },
                "Physical Activity Level": {
                    "sdtype": "numerical"
                },
                "Stress Level": {
                    "sdtype": "categorical"
                },
                "BMI Category": {
                    "sdtype": "categorical"
                },
                "Heart Rate": {
                    "sdtype": "numerical"
                },
                "Daily Steps": {
                    "sdtype": "numerical"


In [ ]:
synthesizer = CTGANSynthesizer(
    metadata,
    epochs=300,
    batch_size=100,
    verbose=True
)

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [ ]:
synthesizer.fit(ctgan_train_data)

Gen. (-01.22) | Discrim. (-01.27): 100%|██████████| 300/300 [00:53<00:00,  5.58it/s]


In [ ]:
from sdv.sampling import Condition

conditions = [
    Condition(
        num_rows=1006,
        column_values={'Sleep Disorder': 'Healthy'}
    ),
    Condition(
        num_rows=244,
        column_values={'Sleep Disorder': 'Sleep Apnea'}
    ),
    Condition(
        num_rows=250,
        column_values={'Sleep Disorder': 'Insomnia'}
    )
]

print(conditions)

[<sdv.sampling.tabular.Condition object at 0x7d26a1b26330>, <sdv.sampling.tabular.Condition object at 0x7d26a1b24920>, <sdv.sampling.tabular.Condition object at 0x7d26a1b25490>]


In [ ]:
synthetic_ctgan = synthesizer.sample_from_conditions(
    conditions=conditions
)

print(synthetic_ctgan.shape)

Sampling conditions: 100%|██████████| 1500/1500 [00:01<00:00, 855.77it/s]

(1500, 13)


In [ ]:
print(synthetic_ctgan['Sleep Disorder'].value_counts())

Sleep Disorder
Healthy        1006
Insomnia        250
Sleep Apnea     244
Name: count, dtype: int64


In [ ]:
synthetic_ctgan['Blood Pressure'] = (
    synthetic_ctgan['Systolic Pressure'].astype(str)
    + '/'
    + synthetic_ctgan['Diastolic Pressure'].astype(str)
)

In [ ]:
synthetic_ctgan.drop(
    ['Systolic Pressure', 'Diastolic Pressure'],
    axis=1,
    inplace=True
)

In [ ]:
synthetic_ctgan.insert(
    0,
    'Person ID',
    range(1, len(synthetic_ctgan) + 1)
)

In [ ]:
print(synthetic_ctgan.shape)
synthetic_ctgan.head()

(1500, 13)


,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder,Blood Pressure
0,1,Male,42,Nurse,8.4,7,63,8,Normal,70,7872,Healthy,120/80
1,2,Male,27,Salesperson,7.3,7,79,6,Normal,71,5864,Healthy,115/80
2,3,Male,47,Accountant,7.6,8,80,8,Normal,71,5707,Healthy,115/81
3,4,Male,32,Engineer,7.5,8,57,6,Normal,80,8076,Healthy,122/79
4,5,Male,45,Teacher,7.5,7,41,3,Normal Weight,66,5554,Healthy,121/87


In [ ]:
synthetic_ctgan.to_csv(
    "CTGAN_1500_Synthetic_Dataset.csv",
    index=False
)

In [ ]:
import os

print(os.path.exists("CTGAN_1500_Synthetic_Dataset.csv"))

True


In [ ]:
from google.colab import files

files.download("CTGAN_1500_Synthetic_Dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>